# Social Media Models — Training

Trains two models from `warehouse.fact_social_media_ml`:

1. **Engagement tier classifier** — Will a post get `Low / Medium / High` engagement?
2. **Donation conversion classifier** — Will a post lead to donations? (`0 / 1`)

Outputs: `engagement_model.sav`, `donation_model.sav` + metadata + metrics JSON files.

## 1. Problem Framing

### Business Problem
Lucero uses social media (Facebook, Instagram, TikTok, and others) to raise awareness, attract donors, and recruit volunteers. The organization's communications team posts content across platforms without a systematic understanding of what types of posts actually drive engagement — and more importantly, which posts lead to tangible fundraising outcomes (donation referrals).

This pipeline addresses two linked predictive questions:
1. **Engagement tier:** Given the characteristics of a planned post (platform, content type, timing, topic), will it generate Low, Medium, or High engagement relative to other posts?
2. **Donation conversion:** Will this post lead to at least one recorded donation referral? (binary: 0/1)

### Who Cares
- **The communications team** needs a content recommendation tool: before publishing a post, they want to know whether the content mix they've chosen is likely to drive engagement or donations.
- **Program leadership** wants to understand which content types (resident stories, program updates, appeals) are most cost-effective at converting social reach into funding.
- **The admin dashboard** can surface these predictions as a "content quality score" for scheduled posts.

### Approach: Predictive
Both models are **predictive**. The goal is to correctly classify a post's likely engagement tier and donation-driving potential before publishing, enabling the team to optimize content before it goes live.

**Why predictive, not explanatory?**  
The team doesn't need to understand *why* certain content performs better in a causal sense — they need an actionable signal ("this post will likely underperform; consider adding a resident story"). A Random Forest that accurately classifies engagement tier from post metadata provides that directly. Causal claims would require A/B testing with random post assignment, which is not feasible.

### Success Metrics
- **Engagement model:** Weighted F1 across Low/Medium/High tiers
- **Conversion model:** Weighted F1; recall on has_donations=1 (conversion events are rare and valuable)
- Deployment: predictions served via API to a "Post Optimizer" UI in the admin panel

In [1]:
import sys
sys.path.insert(0, '../pipeline/jobs')

import json
from datetime import datetime, timezone

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from config import (
    ARTIFACTS_DIR,
    ENGAGEMENT_MODEL_PATH, ENGAGEMENT_METADATA_PATH, ENGAGEMENT_METRICS_PATH,
    DONATION_MODEL_PATH, DONATION_METADATA_PATH, DONATION_METRICS_PATH,
    WAREHOUSE_SCHEMA,
)
from utils_db import get_engine

print('Imports loaded.')

Imports loaded.


## 2. Data Acquisition & Preparation

Data comes from `warehouse.fact_social_media_ml`, built by `etl.py: build_social_warehouse()`. It joins:

| Source | Features |
|---|---|
| `social_media_posts` | Platform, post type, media type, timing, caption text metadata |
| `post_metrics` | Reach, likes, comments, shares, saves → computed into engagement rate |
| `donation_referrals` | Whether the post led to a recorded donation referral |
| `campaigns` | Whether the post was associated with a named campaign |

**Target construction:**
- `engagement_tier`: computed in ETL by splitting `engagement_rate` into terciles (Low / Medium / High). **Important:** the terciles are computed on the full dataset in ETL, which creates a mild data leakage risk — the test set's tier boundaries are influenced by test data. This is acknowledged as a limitation.
- `has_donations`: 1 if `donation_referrals > 0` for this post, 0 otherwise. This is a very sparse binary label (most posts don't directly generate a traceable referral).

**Feature engineering:**
- Categorical features (platform, post type, media type, day of week, CTA type, content topic, sentiment tone) are ordinal-encoded with fixed category lists — unknown values map to -1.
- `post_hour`: hour of day extracted from post timestamp
- `caption_length`: character count of caption (proxy for content depth)
- `is_boosted` + `boost_budget_php`: paid promotion indicators

## Load data

In [2]:
engine = get_engine(WAREHOUSE_SCHEMA)
df = pd.read_sql('SELECT * FROM warehouse.fact_social_media_ml', engine)
print(f'Loaded {len(df)} rows')

print(f'\nEngagement tier distribution:\n{df["engagement_tier"].value_counts().to_string()}')
print(f'\nHas donations distribution:\n{df["has_donations"].value_counts().to_string()}')

Loaded 812 rows

Engagement tier distribution:
engagement_tier
High      271
Low       271
Medium    270

Has donations distribution:
has_donations
1    522
0    290


## 3. Exploratory Data Analysis

We examine engagement tier and donation conversion distributions, posting patterns by platform and time, and feature relationships before modeling.

In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Engagement tier distribution
if 'engagement_tier' in df.columns:
    et = df['engagement_tier'].value_counts()
    axes[0,0].bar(et.index, et.values, color=['#EF5350','#FFA726','#66BB6A'])
    axes[0,0].set_title('Engagement Tier Distribution', fontweight='bold')
    for i, v in enumerate(et.values):
        axes[0,0].text(i, v + 0.3, str(v), ha='center')

# 2. Donation conversion distribution
if 'has_donations' in df.columns:
    dc = df['has_donations'].value_counts()
    axes[0,1].bar(['No Donation (0)', 'Has Donation (1)'], [dc.get(0,0), dc.get(1,0)],
                  color=['#90A4AE','#26A69A'])
    axes[0,1].set_title('Donation Conversion Distribution', fontweight='bold')
    axes[0,1].set_ylabel('Post Count')
    for i, v in enumerate([dc.get(0,0), dc.get(1,0)]):
        axes[0,1].text(i, v + 0.2, str(v), ha='center')

# 3. Engagement by platform
if 'platform_encoded' in df.columns and 'engagement_tier' in df.columns:
    plat_eng = df.groupby('platform_encoded')['engagement_tier'].value_counts(normalize=True).unstack(fill_value=0)
    plat_eng.plot(kind='bar', stacked=True, ax=axes[1,0],
                  color=['#EF5350','#FFA726','#66BB6A'])
    axes[1,0].set_title('Engagement Tier by Platform (encoded)', fontweight='bold')
    axes[1,0].set_xlabel('Platform (encoded)')
    axes[1,0].legend(title='Tier', fontsize=8)
    axes[1,0].tick_params(axis='x', rotation=0)

# 4. Post hour vs engagement
if 'post_hour' in df.columns and 'engagement_tier' in df.columns:
    df_hour = df.copy()
    tier_map = {'Low': 0, 'Medium': 1, 'High': 2}
    df_hour['tier_num'] = df_hour['engagement_tier'].map(tier_map)
    hour_eng = df_hour.groupby('post_hour')['tier_num'].mean()
    axes[1,1].plot(hour_eng.index, hour_eng.values, marker='o', color='#1565C0')
    axes[1,1].set_title('Average Engagement Score by Hour of Day', fontweight='bold')
    axes[1,1].set_xlabel('Hour of Post (0-23)')
    axes[1,1].set_ylabel('Avg Tier (0=Low, 1=Med, 2=High)')
    axes[1,1].set_xticks(range(0, 24, 2))

plt.suptitle('Social Media EDA', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_social_media.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Dataset shape: {df.shape}")
print(f"Donation conversion rate: {df['has_donations'].mean():.1%}" if 'has_donations' in df.columns else "")

Dataset shape: (812, 19)
Donation conversion rate: 64.3%


/var/folders/cn/qpbmvkhx6w38khn86x0k8cdh0000gn/T/ipykernel_27150/2941575742.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Feature columns

In [4]:
CAT_COLS = [
    'platform_encoded', 'post_type_encoded', 'media_type_encoded',
    'day_of_week_encoded', 'call_to_action_type_encoded',
    'content_topic_encoded', 'sentiment_tone_encoded',
]
NUM_COLS = [
    'has_call_to_action', 'features_resident_story', 'is_boosted',
    'post_hour', 'num_hashtags', 'mentions_count', 'caption_length',
    'boost_budget_php', 'follower_count_at_post',
]
FEATURE_COLS = CAT_COLS + NUM_COLS

CAT_CATEGORIES = [
    [0,1,2,3,4,5,6], [0,1,2,3,4,5], [0,1,2,3,4],
    [0,1,2,3,4,5,6], [0,1,2,3,4], [0,1,2,3,4,5,6,7,8], [0,1,2,3,4,5],
]

def make_pipeline():
    cat_transformer = Pipeline([('encoder', OrdinalEncoder(
        categories=CAT_CATEGORIES, handle_unknown='use_encoded_value', unknown_value=-1
    ))])
    num_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    preprocessor = ColumnTransformer([
        ('cat', cat_transformer, CAT_COLS),
        ('num', num_transformer, NUM_COLS),
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')),
    ])

X = df[FEATURE_COLS]
print(f'Feature matrix: {X.shape}')

Feature matrix: (812, 16)


## 4. Modeling & Feature Selection

**Algorithm:** Random Forest Classifier (both models)

Both models use the same `make_pipeline()` factory that applies a `ColumnTransformer` with:
- `OrdinalEncoder` for categorical columns (platform, content type, etc.) with fixed category lists so that new posts with unseen values map to -1 rather than crashing
- `SimpleImputer` + `StandardScaler` for numeric columns

`class_weight='balanced'` is critical for the donation conversion model because the positive class (posts that generate referrals) is rare — without it, the model would predict 0 for nearly every post.

**Feature selection rationale:**
- **Categorical features** (platform, post type, media type, day, CTA, topic, sentiment): these capture the content strategy dimensions the team can actually control
- **Numeric features** (hour, hashtag count, caption length, budget, follower count): operational levers and context
- Features excluded: raw engagement counts (likes, shares) would leak the target since engagement rate is derived from them

## Model 1 — Engagement tier classifier

In [5]:
y_eng = df['engagement_tier']
min_class = y_eng.value_counts().min()
stratify  = y_eng if min_class >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y_eng, test_size=0.25, random_state=42, stratify=stratify
)

eng_model = make_pipeline()
eng_model.fit(X_train, y_train)

y_pred   = eng_model.predict(X_test)
accuracy = float(accuracy_score(y_test, y_pred))
f1       = float(f1_score(y_test, y_pred, average='weighted'))
report   = classification_report(y_test, y_pred, output_dict=True)

print(f'Engagement — Accuracy: {accuracy:.3f} | Weighted F1: {f1:.3f}')
print(classification_report(y_test, y_pred))

Engagement — Accuracy: 0.626 | Weighted F1: 0.625
              precision    recall  f1-score   support

        High       0.61      0.78      0.68        68
         Low       0.83      0.66      0.74        68
      Medium       0.47      0.43      0.45        67

    accuracy                           0.63       203
   macro avg       0.64      0.62      0.62       203
weighted avg       0.64      0.63      0.62       203



In [6]:
# Feature importance — engagement tier model
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

rf_eng = eng_model.named_steps['clf']
feat_imp_eng = pd.Series(rf_eng.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp_eng.sort_values().plot(kind='barh', ax=ax, color='#1565C0')
ax.set_title('Feature Importances — Social Media Engagement Tier Classifier', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.savefig('feature_importance_engagement.png', dpi=100, bbox_inches='tight')
plt.show()
print("Top features (engagement):")
print(feat_imp_eng.head(8).to_string())

Top features (engagement):
post_hour                 0.303967
sentiment_tone_encoded    0.101456
caption_length            0.095529
follower_count_at_post    0.075547
content_topic_encoded     0.061048
day_of_week_encoded       0.054284
num_hashtags              0.047939
post_type_encoded         0.045417


/var/folders/cn/qpbmvkhx6w38khn86x0k8cdh0000gn/T/ipykernel_27150/2143252480.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Engagement Model — Business Interpretation

**Interpreting the metrics:**
- A weighted F1 above 0.65 indicates the model can meaningfully distinguish High-engagement posts from Low ones, enabling the team to optimize content strategy.
- The classification report's **High class recall** is most important: can the model identify truly high-performing posts? High false negatives here mean missed viral opportunities.

**What the feature importances tell us operationally:**
- Features with high importance that the team can control (content topic, post type, timing) are the levers to pull.
- Features reflecting platform context (follower count at post time) are less actionable but help contextualize performance.

**Practical use:** Before scheduling a post, a social worker or comms team member can input the planned post characteristics into the Post Optimizer UI and receive a predicted engagement tier and donation conversion probability.

In [7]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(eng_model, str(ENGAGEMENT_MODEL_PATH))
with open(ENGAGEMENT_METADATA_PATH, 'w') as f:
    json.dump({'model_name': 'social_media_engagement_classifier', 'model_version': '1.0.0',
               'trained_at_utc': datetime.now(timezone.utc).isoformat(),
               'warehouse_table': 'fact_social_media_ml',
               'num_training_rows': int(len(X_train)), 'num_test_rows': int(len(X_test)),
               'label_col': 'engagement_tier', 'feature_cols': FEATURE_COLS,
               'cat_cols': CAT_COLS, 'num_cols': NUM_COLS,
               'classes': list(eng_model.classes_)}, f, indent=2)
with open(ENGAGEMENT_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy, 'weighted_f1': f1, 'classification_report': report}, f, indent=2)
print(f'Saved: {ENGAGEMENT_MODEL_PATH}')

Saved: /Users/samjenson/Desktop/INTEX_II/pipeline/artifacts/engagement_model.sav


## Model 2 — Donation conversion classifier

In [8]:
y_don = df['has_donations'].astype(int)
min_class = y_don.value_counts().min()
stratify  = y_don if min_class >= 2 else None

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X, y_don, test_size=0.25, random_state=42, stratify=stratify
)

don_model = make_pipeline()
don_model.fit(X_train_d, y_train_d)

y_pred_d   = don_model.predict(X_test_d)
accuracy_d = float(accuracy_score(y_test_d, y_pred_d))
f1_d       = float(f1_score(y_test_d, y_pred_d, average='weighted'))
report_d   = classification_report(y_test_d, y_pred_d, output_dict=True)

print(f'Donation — Accuracy: {accuracy_d:.3f} | Weighted F1: {f1_d:.3f}')
print(classification_report(y_test_d, y_pred_d))

Donation — Accuracy: 0.803 | Weighted F1: 0.800
              precision    recall  f1-score   support

           0       0.75      0.67      0.71        73
           1       0.83      0.88      0.85       130

    accuracy                           0.80       203
   macro avg       0.79      0.77      0.78       203
weighted avg       0.80      0.80      0.80       203



In [9]:
# Feature importance — donation conversion model
rf_don = don_model.named_steps['clf']
feat_imp_don = pd.Series(rf_don.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp_don.sort_values().plot(kind='barh', ax=ax, color='#2E7D32')
ax.set_title('Feature Importances — Donation Conversion Classifier', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.savefig('feature_importance_donation.png', dpi=100, bbox_inches='tight')
plt.show()
print("Top features (donation conversion):")
print(feat_imp_don.head(8).to_string())

Top features (donation conversion):
post_hour                  0.195225
caption_length             0.126534
post_type_encoded          0.119342
features_resident_story    0.102544
follower_count_at_post     0.066964
sentiment_tone_encoded     0.063707
content_topic_encoded      0.048564
day_of_week_encoded        0.047066


/var/folders/cn/qpbmvkhx6w38khn86x0k8cdh0000gn/T/ipykernel_27150/1532499932.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
joblib.dump(don_model, str(DONATION_MODEL_PATH))
with open(DONATION_METADATA_PATH, 'w') as f:
    json.dump({'model_name': 'social_media_donation_classifier', 'model_version': '1.0.0',
               'trained_at_utc': datetime.now(timezone.utc).isoformat(),
               'warehouse_table': 'fact_social_media_ml',
               'num_training_rows': int(len(X_train_d)), 'num_test_rows': int(len(X_test_d)),
               'label_col': 'has_donations', 'feature_cols': FEATURE_COLS,
               'cat_cols': CAT_COLS, 'num_cols': NUM_COLS,
               'classes': [int(c) for c in don_model.classes_]}, f, indent=2)
with open(DONATION_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy_d, 'weighted_f1': f1_d, 'classification_report': report_d}, f, indent=2)
print(f'Saved: {DONATION_MODEL_PATH}')

Saved: /Users/samjenson/Desktop/INTEX_II/pipeline/artifacts/donation_model.sav


### Cross-Validation — Both Social Media Models

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_eng = cross_val_score(eng_model, X, y_eng, cv=cv, scoring='f1_weighted', n_jobs=-1)
print("── Engagement Tier Model ────────────────────────")
print(f"5-Fold CV Weighted F1:  {cv_eng.mean():.3f} ± {cv_eng.std():.3f}")
print(f"Individual fold scores: {[round(s, 3) for s in cv_eng]}")
print(f"Holdout test F1:        {f1:.3f}")

cv_don = cross_val_score(don_model, X, y_don, cv=cv, scoring='f1_weighted', n_jobs=-1)
print("\n── Donation Conversion Model ────────────────────")
print(f"5-Fold CV Weighted F1:  {cv_don.mean():.3f} ± {cv_don.std():.3f}")
print(f"Individual fold scores: {[round(s, 3) for s in cv_don]}")
print(f"Holdout test F1:        {f1_d:.3f}")

── Engagement Tier Model ────────────────────────
5-Fold CV Weighted F1:  0.646 ± 0.014
Individual fold scores: [np.float64(0.646), np.float64(0.665), np.float64(0.654), np.float64(0.642), np.float64(0.623)]
Holdout test F1:        0.625



── Donation Conversion Model ────────────────────
5-Fold CV Weighted F1:  0.779 ± 0.033
Individual fold scores: [np.float64(0.729), np.float64(0.813), np.float64(0.803), np.float64(0.799), np.float64(0.75)]
Holdout test F1:        0.800


## 5. Causal & Relationship Analysis

### What the Models Reveal About Social Media Performance

**Engagement predictors:**
The most important features for engagement tier prediction are expected to include `content_topic_encoded`, `post_type_encoded`, and timing variables (`post_hour`, `day_of_week_encoded`). This aligns with general social media research: content relevance and optimal posting time strongly predict engagement.

`features_resident_story` (whether the post features a resident's story) is likely a strong positive predictor of both engagement and donation conversion — personal narratives drive emotional connection and giving behavior.

`is_boosted` and `boost_budget_php` are important controls: boosted posts get artificial reach that inflates engagement metrics, so the model implicitly learns that paid reach is a strong predictor of measured engagement.

**Donation conversion predictors:**
Donation conversion is a much noisier signal. The features most predictive of conversion likely include `has_call_to_action` and `call_to_action_type_encoded` — posts with explicit donation asks are unsurprisingly more likely to generate referrals.

### Critical Limitations and Causal Caveats

1. **Engagement tier is self-referential:** The target (`engagement_tier`) is computed from engagement rate, which aggregates the same post-level metrics (reach, likes, etc.) that temporal patterns in the feature set already capture. The model may be recovering data artifacts rather than genuine signal.

2. **Donation attribution is noisy:** The `donation_referrals` field counts referrals that were *tracked* to a specific post — but many donations inspired by a post may be made directly on the website without a trackable referral link. The true conversion rate is likely higher, and the current label is a lower-bound noisy proxy.

3. **Boosting confounds organic performance:** Paid-boosted posts have higher measured engagement not because the content is better, but because they were shown to more people. Any model that includes `is_boosted` must be interpreted with this in mind — the coefficient on boost budget cannot be interpreted as "spending more money causes engagement" without controlling for audience size.

4. **Platform algorithm changes:** Social media engagement dynamics change rapidly as platforms update their algorithms. A model trained on 6-month-old data may already be partially stale.

### What Can Be Acted On Despite These Limitations
- **Content strategy:** If `features_resident_story` and specific `content_topic_encoded` categories are consistently high-importance, the team can prioritize those content types with confidence.
- **Timing optimization:** If certain hours or days systematically predict higher engagement, the scheduling tool can default to those windows.
- **A/B testing recommendation:** To move from prediction to causation, the team should run structured A/B tests: post similar content at different hours, with and without CTAs, to establish *causal* impact of specific content decisions.

## 6. Deployment Notes

### Integration
1. **Training:** `pipeline/jobs/train_social.py` — trains both models, writes `.sav` and metadata JSON to `pipeline/artifacts/`
2. **Inference:** `pipeline/jobs/inference_social.py` — scores historical and scheduled posts, writes to `operational.social_media_predictions` (columns: `post_id`, `predicted_engagement_tier`, `predicted_has_donations`, `run_at`)
3. **Application integration:** The predicted engagement tier and donation conversion probability are surfaced in the admin panel's Social Media section, providing the team with a "content quality score" before publishing.

### Post Optimizer UI (planned)
A future enhancement is a form-based interface in the admin dashboard where the comms team inputs planned post characteristics (platform, content type, timing, whether it features a resident story, etc.) and receives a real-time engagement tier prediction and donation conversion probability. This is the highest-value deployment form for this model.

### Pipeline Commands
```bash
python pipeline/jobs/etl.py             # rebuild fact_social_media_ml
python pipeline/jobs/train_social.py    # retrain → artifacts/
python pipeline/jobs/inference_social.py  # score → DB
```

### Re-training Cadence
Monthly minimum; more frequently if the organization makes significant changes to content strategy or if a platform undergoes an algorithm update.